# 01 — Tải & chuẩn bị dữ liệu (US train + VN test)

Chạy được nhiều lần: `--resume` / bỏ qua ảnh đã tải. Mục tiêu đề tài: **US 50k–150k**,
**VN 2k–5k** (đã kiểm tra tay một phần).

In [ ]:
# Mỗi notebook là một phiên Colab riêng: mount Drive + đặt DATA_ROOT + vào repo (đã clone ở 00_setup)
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["DATA_ROOT"] = "/content/drive/MyDrive/trademark_data"
%cd /content/major_proj_1

## US (USPTO): metadata → ảnh → lọc → chia tập → LMDB

In [ ]:
!bash scripts/01_get_meta.sh --limit 60000 --resume
!bash scripts/02_get_images.sh --limit 60000 --workers 16
!python -m trademark_ocr.fetch_images --verify

In [ ]:
!python -m trademark_ocr.clean
!python -m trademark_ocr.split --seed 1111
!for s in train val test_us; do python -m trademark_ocr.build_lmdb --subset $s; done

## VN (WIPO Publish): crawl → duyệt tay → LMDB

Sau `make`: tải `processed/review_vn.csv` từ Drive, mở `review_vn.html` để xem ảnh, sửa cột
`visible_text` cho đúng chữ trong ảnh (để trống = bỏ mẫu), tải lên lại rồi chạy `apply`.
`label_noise_rate` trong `review_vn_report.json` cần nêu trong báo cáo.

In [ ]:
!bash scripts/03_get_vn.sh --limit 2000
!python -m trademark_ocr.vn_review make --n 300

In [ ]:
!python -m trademark_ocr.vn_review apply
!python -m trademark_ocr.build_lmdb --subset test_vn --split-csv $DATA_ROOT/processed/split_test_vn_verified.csv